# Phase 3 — The core 2³ factorial + interaction analysis

Completes the cube: Phase 2 measured the four single-factor corners (baseline, W, K, S) at
repeat 0; this runbook adds the four interaction corners (**W+S, K+S, W+K+S, W+K**) and then
the repeats. **All cells on A100** (IMPLEMENTATION_PLAN Phase 3, GPU-confound correction).

Split into three independently resumable sessions, ordered so a dead session still leaves
maximally interpretable data:

| session | cells | est. units (real Phase-2 rate: 11.73 u / 48 cells ≈ 0.24 u/cell) |
|---|---|---|
| A — interaction corners @ r0 | 48 (4 launches) | ~12–15 → completes all 8 corners: first full interaction matrix (point estimates) |
| B — repeat 1, all 8 corners | 96 (8 launches) | ~25–30 → 2-repeat spread on every effect |
| C — repeat 2, all 8 corners | 96 (8 launches) | ~25–30 → full 3-repeat spread |

Session-A corner order is deliberate (see the marginals data, `phase2_results/`):
**ws first** (headline W×S cell — both factors erode to ~1.0x by conc 64 alone, and this
pairing is the one GPU-untested combination), **ks second** (the genuinely-uncertain
QuantSpec cell: tau under quantized KV — Phase 2 showed tau is flat in concurrency, so any
tau movement here is attributable to KV quant), **wks third** (completes the interference
gap), **wk last** (predicted near-additive control).

**Requirements:** A100 runtime (verify below), `HF_TOKEN` Colab secret with Llama-3.1
access. Expect vLLM's FP8-KV "accuracy drop without proper scaling factor" warning on
K-corners — known and documented (PREREQ_RESULTS Check 6).

In [ ]:
# 1. Verify the GPU actually assigned
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
units_before = input("Compute-units balance right now: ")

In [ ]:
# 2. Repo + Spec-Bench (RAG passages; external/ is git-ignored)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK" 

In [ ]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)" 

In [ ]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 5. Harness self-check (~1 min, no GPU). Includes the verified-against-
#    synthetic-cubes factorial statistics (tests/test_factorial.py).
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

In [ ]:
# 6. Sanity: session-A server commands (nothing launches)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_ws_*_r0.yaml" "configs/factorial/cube_ks_*_r0.yaml" "configs/factorial/cube_wks_*_r0.yaml" "configs/factorial/cube_wk_*_r0.yaml" --results-dir results --dry-run 2>&1 | grep "server command" 

In [ ]:
# 7. SESSION A: interaction corners at repeat 0 (ws -> ks -> wks -> wk).
# Resumable; a disconnect costs one cell. ~12-15 units.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_ws_*_r0.yaml" "configs/factorial/cube_ks_*_r0.yaml" "configs/factorial/cube_wks_*_r0.yaml" "configs/factorial/cube_wk_*_r0.yaml" --results-dir results

In [ ]:
# 8. First full interaction matrix (point estimates). NOTE: the analysis
# needs the Phase-2 marginal records in the same results dir -- if this is a
# fresh session, restore them first (unzip phase2_results.zip into results/,
# or copy results/runs/*.json from the committed phase2_results/runs/).
!cp -n phase2_results/runs/*.json results/runs/ 2>/dev/null; echo "phase2 records merged"
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.factorial results

In [ ]:
# 9. SESSION B: repeat 1 for ALL 8 corners (96 cells, 8 launches, ~25-30
# units). Fine to run in a separate Colab session -- re-run cells 1-5 first.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_*_r1.yaml" --results-dir results

In [ ]:
# 10. SESSION C: repeat 2 for ALL 8 corners (96 cells, ~25-30 units).
# Kill lever if units run short: skip C at conc 1 and 8 but KEEP conc 32 and
# 64 -- the crossovers live between 32 and 64 (W hits 1.01x, S hits 0.97x on
# GSM8K at c64), so that's where spread decides whether a sign claim holds.
# Selective globs for that: "configs/factorial/cube_*_c32_r2.yaml" "configs/factorial/cube_*_c64_r2.yaml"
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_*_r2.yaml" --results-dir results

In [ ]:
# 11. Final analysis: effects with [min..max] spread across repeats,
# interference gap per workload x concurrency, machine-readable JSON.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.factorial results
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.marginals results

### Session D — K-stress addendum (~20–25 units, 16 cells, 2 launches)

Phase 2's K marginal was ~flat — a real null for its regime: KV demand peaked at ~18% of
the pool, so K's capacity channel never engaged. Session D creates the pressure: unique
~7.4k-token documents (`prefix_overlap: low` — nothing shareable), FP16 weights, no spec,
{FP16-KV, FP8-KV} × concurrency {32, 48, 64, 96} × 2 repeats.

Predicted (Llama-3.1-8B: 128 KiB/token FP16-KV, halved by FP8): on an **80GB** A100 the
FP16 ceiling is ~47 concurrent requests (conc 48 sits right on it); on a **40GB** card
it's ~15 and everything past conc 32 diverges. Either card works — note which one cell 1
reported. After the sweep, grep the FP16 server log for `GPU KV cache size` and pass the
token count to the analysis for the predicted-vs-measured plateau comparison.

In [ ]:
# D1. K-stress sweep (resumable like everything else)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/k_stress/kstress_*.yaml" --results-dir results

In [ ]:
# D2. Pool size from the FP16-KV server log -> predicted plateau in the report
!grep -h "GPU KV cache size" results/server_logs/*.log | tail -2
POOL = input("FP16-KV pool size in tokens (from the line above, FP16 group): ").replace(",", "")
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.k_stress results --pool-tokens $POOL

**Reading Session D:** the divergence signature is (1) FP16-KV `kv-usage max` pinned at
~1.00 with preemptions > 0 while FP8-KV sits near ~0.5 with none — proof the ceiling was
hit, not inferred; (2) FP16 emergent batch flat at the predicted plateau while FP8 tracks
concurrency; (3) the goodput ratio climbing above ~1.5x past the ceiling. Below the
ceiling (conc 32) expect ratio ≈ 0.95–1.05 — that's the Phase-2 null reproducing, which
is what makes the two-regime story credible. This feeds the decision guide as: *enable
FP8-KV when projected demand (concurrency × context × 128 KiB) approaches the pool; below
that it costs ~5% on A100 and buys nothing.*

In [ ]:
# 12. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase3_results.zip results
from google.colab import files
files.download("phase3_results.zip")

## Reading the result

The factorial report is the project's headline artifact. Per workload × concurrency:

- **Interference gap**: naive speedup (product of the three marginal wins) vs measured
  full-stack speedup, and the overestimate factor. Phase-2 marginals predict the naive
  product at conc=1 is ~4.2–6.4x depending on workload — the gap line says how much of
  that survives stacking, and the trend across concurrency says whether interference
  grows with load (the thesis).
- **W×S contrast** (headline): negative = W and S cannibalize each other (Block 0's
  batch-1 finding, now with a concurrency axis). Compare S-on-W relative speedup vs
  Phase 2's S-on-FP16 column at each concurrency.
- **K×S contrast + tau in the ks/wks cells**: Phase 2 measured tau flat in concurrency
  (2.85/4.08/2.52 by workload). If tau moves in the K-on cells, that's the QuantSpec
  acceptance channel showing up (or failing to) under continuous batching — either way
  a reportable finding.
- **W×K contrast**: the pre-registered "boring control" — expect ~0 ('~0?' flag). If it
  isn't, inspect before celebrating: both marginals were near-null at c64, so a large
  W×K would smell like a harness artifact.
- **'~0?' flags**: any effect whose across-repeat spread straddles zero is not a claim.
  After session B you have 2-repeat spreads; after C, 3.

Afterwards: commit results + reports, update PREREQ_RESULTS Check 1 with this session's
burn, and what remains is Phase 4 (SGLang seam, optional) and Phase 5 (decision guide +
write-ups).